# LeetCode 1708 — Largest Subarray Length K bằng Python

**Link đề:** https://leetcode.com/problems/largest-subarray-length-k/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def largestSubarray(self, nums: List[int], k: int) -> List[int]:
        ...
```

## Mục tiêu học
1. Hiểu phép **so sánh từ điển (lexicographic)** giữa hai mảng và biết Python so sánh `list` đúng theo cách đó.
2. Liệt kê mọi **subarray (mảng con liên tiếp)** dài `k` bằng slicing, và đếm có `n - k + 1` mảng con.
3. Brute force `O((n-k+1)·k)` → nhận ra **phần tử đầu tiên quyết định tất cả** khi các số **phân biệt** → lời giải tham lam `O(n)`.
4. Hiểu vì sao chỉ được chọn điểm bắt đầu trong `[0, n - k]`.
5. Mở rộng sang câu hỏi follow-up: nếu các số **không** phân biệt thì sao?

## 1. Đề bài và ví dụ

Mảng `A` **lớn hơn** mảng `B` (cùng độ dài) nếu ở **chỉ số đầu tiên** `i` mà `A[i] != B[i]`, ta có `A[i] > B[i]`.
- `[1,3,2,4] > [1,2,2,4]` vì ở chỉ số 1: `3 > 2`.
- `[1,4,4,4] < [2,1,1,1]` vì ở chỉ số 0: `1 < 2`.

Một **subarray** là dãy **liên tiếp** các phần tử của mảng. Cho mảng `nums` gồm các số nguyên **phân biệt** và số `k`, trả về **subarray dài `k` lớn nhất**.

| `nums` | `k` | Các subarray dài k | Kết quả |
|---|:---:|---|---|
| `[1,4,5,2,3]` | 3 | `[1,4,5]`, `[4,5,2]`, `[5,2,3]` | `[5,2,3]` |
| `[1,4,5,2,3]` | 4 | `[1,4,5,2]`, `[4,5,2,3]` | `[4,5,2,3]` |
| `[1,4,5,2,3]` | 1 | `[1]`,`[4]`,`[5]`,`[2]`,`[3]` | `[5]` |

**Constraints (theo đề gốc):** `1 <= k <= len(nums) <= 10^5`; `1 <= nums[i] <= 10^9`; mọi phần tử của `nums` **khác nhau**.

**Follow-up:** nếu các số trong `nums` **không** phân biệt thì sao?

### 1.1. So sánh hai mảng bằng tay

Giống tra từ điển: so chữ cái đầu, bằng nhau thì so chữ cái thứ hai… Chỉ vị trí **khác nhau đầu tiên** mới quyết định; mọi thứ sau đó không quan trọng.

In [1]:
from typing import List

def first_diff(A, B):
    for i in range(len(A)):
        if A[i] != B[i]:
            return i
    return None

for A, B_ in [([1, 3, 2, 4], [1, 2, 2, 4]), ([1, 4, 4, 4], [2, 1, 1, 1])]:
    i = first_diff(A, B_)
    print(A, "vs", B_, "-> khác đầu tiên ở i =", i, "->", "A lớn hơn" if A[i] > B_[i] else "B lớn hơn")

[1, 3, 2, 4] vs [1, 2, 2, 4] -> khác đầu tiên ở i = 1 -> A lớn hơn
[1, 4, 4, 4] vs [2, 1, 1, 1] -> khác đầu tiên ở i = 0 -> B lớn hơn


## 2. Nền tảng Python cần biết

### 2.1. Slicing để lấy subarray

`nums[i:i+k]` là subarray dài `k` bắt đầu ở `i`. Điểm bắt đầu hợp lệ: `i = 0, 1, ..., n - k` ⇒ có **`n - k + 1`** subarray. Nếu `i > n - k` thì lát cắt **ngắn hơn `k`** (Python không báo lỗi!).

In [2]:
nums = [1, 4, 5, 2, 3]
k = 3
n = len(nums)
for i in range(n - k + 1):
    print("i =", i, "->", nums[i:i + k])
print("i = 3 (không hợp lệ) ->", nums[3:3 + k], "chỉ dài", len(nums[3:3 + k]))

i = 0 -> [1, 4, 5]
i = 1 -> [4, 5, 2]
i = 2 -> [5, 2, 3]
i = 3 (không hợp lệ) -> [2, 3] chỉ dài 2


### 2.2. Python so sánh `list` theo thứ tự từ điển

Toán tử `<`, `>`, `max()` trên list dùng **đúng** định nghĩa của đề (với hai list cùng độ dài). Đây là công cụ cực tiện cho brute force.

In [3]:
print([1, 3, 2, 4] > [1, 2, 2, 4])
print([1, 4, 4, 4] < [2, 1, 1, 1])
print(max([[1, 4, 5], [4, 5, 2], [5, 2, 3]]))
print("độ dài khác nhau: [5] > [4, 9, 9] ->", [5] > [4, 9, 9], "| [4] < [4, 0] ->", [4] < [4, 0])

True
True
[5, 2, 3]
độ dài khác nhau: [5] > [4, 9, 9] -> True | [4] < [4, 0] -> True


Lưu ý dòng cuối: khi một list là **tiền tố** của list kia, list **ngắn hơn** nhỏ hơn. Bài này mọi subarray cùng dài `k` nên không gặp trường hợp đó.

### 2.3. `max()` với `range` và tham số `key`

`max(iterable, key=f)` trả về phần tử có `f(x)` lớn nhất (nếu hoà, lấy phần tử **đầu tiên**). Ta có thể tìm **chỉ số** của giá trị lớn nhất: `max(range(m), key=lambda i: nums[i])`.

`lambda i: nums[i]` là **hàm vô danh** một dòng: nhận `i`, trả `nums[i]`.

In [4]:
nums = [1, 4, 5, 2, 3]
best_i = max(range(len(nums)), key=lambda i: nums[i])
print("chỉ số của phần tử lớn nhất:", best_i, "giá trị:", nums[best_i])

square = lambda x: x * x
print("lambda ví dụ:", square(7))

chỉ số của phần tử lớn nhất: 2 giá trị: 5
lambda ví dụ: 49


### 2.4. `list.index(x)` và `max()` trên một lát cắt

`nums.index(x)` trả về chỉ số **đầu tiên** của `x`. Kết hợp: `nums.index(max(nums[:n-k+1]))`. Chú ý `index` tìm từ đầu mảng, vì giá trị phân biệt nên kết quả duy nhất.

In [5]:
nums = [1, 4, 5, 2, 3]
for k in [1, 3, 4, 5]:
    prefix = nums[:len(nums) - k + 1]
    m = max(prefix)
    print(f"k = {k}: vùng bắt đầu hợp lệ = {prefix}, max = {m}, index = {nums.index(m)}")

k = 1: vùng bắt đầu hợp lệ = [1, 4, 5, 2, 3], max = 5, index = 2
k = 3: vùng bắt đầu hợp lệ = [1, 4, 5], max = 5, index = 2
k = 4: vùng bắt đầu hợp lệ = [1, 4], max = 4, index = 1
k = 5: vùng bắt đầu hợp lệ = [1], max = 1, index = 0


## 3. Cách trực giác — Brute force so sánh mọi subarray

1. Viết hàm `is_larger(A, B)` đúng định nghĩa đề.
2. Giữ `best = nums[0:k]`. Với mỗi `i = 1..n-k`, lấy `cand = nums[i:i+k]`; nếu `is_larger(cand, best)` thì `best = cand`.

In [6]:
def is_larger(A: List[int], B: List[int]) -> bool:
    for x, y in zip(A, B):        # zip ghép cặp phần tử cùng vị trí
        if x != y:
            return x > y
    return False                   # bằng nhau hoàn toàn -> không lớn hơn

def largest_brute(nums: List[int], k: int) -> List[int]:
    best = nums[0:k]
    for i in range(1, len(nums) - k + 1):
        cand = nums[i:i + k]
        if is_larger(cand, best):
            best = cand
    return best

print(list(zip([1, 3, 2], [1, 2, 2])))
print(largest_brute([1, 4, 5, 2, 3], 3), largest_brute([1, 4, 5, 2, 3], 4), largest_brute([1, 4, 5, 2, 3], 1))

[(1, 1), (3, 2), (2, 2)]
[5, 2, 3] [4, 5, 2, 3] [5]


`zip(A, B)` tạo các cặp `(A[0], B[0]), (A[1], B[1]), ...` — tiện để duyệt song song hai dãy.

Phiên bản "Pythonic" của brute force dùng `max` trên các lát cắt:

In [7]:
def largest_brute_max(nums: List[int], k: int) -> List[int]:
    return max(nums[i:i + k] for i in range(len(nums) - k + 1))

print(largest_brute_max([1, 4, 5, 2, 3], 3))

[5, 2, 3]


### Chi phí của brute force

Có `n - k + 1` subarray, mỗi lần cắt + so sánh tốn tới `O(k)` ⇒ `O((n - k + 1) · k)`. Tệ nhất khi `k ≈ n/2`: khoảng `n²/4` = `2.5·10^9` với `n = 10^5` — **quá chậm**.

In [8]:
for n in [10, 1000, 100000]:
    k = n // 2
    print(f"n = {n:>6}, k = {k:>5}: số phần tử phải cắt/so sánh ≈ {(n - k + 1) * k:,}")

n =     10, k =     5: số phần tử phải cắt/so sánh ≈ 30
n =   1000, k =   500: số phần tử phải cắt/so sánh ≈ 250,500
n = 100000, k = 50000: số phần tử phải cắt/so sánh ≈ 2,500,050,000


## 4. Insight — phần tử đầu tiên quyết định tất cả

So sánh hai subarray bắt đầu ở `i` và `j` (`i != j`): vị trí đầu tiên là `nums[i]` và `nums[j]`. Vì các số **phân biệt**, `nums[i] != nums[j]` ⇒ **chỉ cần so phần tử đầu**!

⇒ Subarray lớn nhất là subarray bắt đầu tại **phần tử lớn nhất trong vùng bắt đầu hợp lệ** `nums[0 .. n-k]`.

Chú ý: phần tử lớn nhất **cả mảng** có thể nằm quá gần cuối (không đủ `k` phần tử sau nó). Ví dụ `nums = [1, 4, 5, 2, 3]`, `k = 4`: max cả mảng là `5` ở chỉ số 2, nhưng chỉ còn 3 phần tử từ đó → không hợp lệ; vùng hợp lệ là `[1, 4]` → bắt đầu ở `4`.

In [9]:
nums, k = [1, 4, 5, 2, 3], 4
n = len(nums)
print("vùng bắt đầu hợp lệ: chỉ số 0 ..", n - k, "->", nums[:n - k + 1])
print("max cả mảng ở chỉ số", nums.index(max(nums)), "-> lát cắt", nums[nums.index(max(nums)):nums.index(max(nums)) + k], "(thiếu phần tử!)")

vùng bắt đầu hợp lệ: chỉ số 0 .. 1 -> [1, 4]
max cả mảng ở chỉ số 2 -> lát cắt [5, 2, 3] (thiếu phần tử!)


## 5. Lời giải chính — tham lam `O(n)`

**Đoạn 1:** `n = len(nums)`; điểm bắt đầu hợp lệ cuối cùng là `n - k`.
**Đoạn 2:** duyệt `i = 0..n-k`, giữ `best_i` là chỉ số có `nums[i]` lớn nhất.
**Đoạn 3:** trả `nums[best_i : best_i + k]`.

In [10]:
def largest_greedy(nums: List[int], k: int) -> List[int]:
    n = len(nums)
    best_i = 0
    for i in range(1, n - k + 1):         # chỉ xét điểm bắt đầu hợp lệ
        if nums[i] > nums[best_i]:
            best_i = i
    return nums[best_i:best_i + k]

for k in [1, 3, 4, 5]:
    print("k =", k, "->", largest_greedy([1, 4, 5, 2, 3], k))

k = 1 -> [5]
k = 3 -> [5, 2, 3]
k = 4 -> [4, 5, 2, 3]
k = 5 -> [1, 4, 5, 2, 3]


## 6. TRACE từng bước

Ví dụ `nums = [1, 4, 5, 2, 3]`, `k = 3` ⇒ `n - k = 2`, chỉ xét `i = 0, 1, 2`.

| i | nums[i] | nums[best_i] trước | Cập nhật? | best_i sau |
|---:|---:|---:|---|---:|
| 0 | 1 | — | khởi tạo | 0 |
| 1 | 4 | 1 | 4 > 1 → có | 1 |
| 2 | 5 | 4 | 5 > 4 → có | 2 |
| 3 | 2 | — | **không xét** (3 > n − k) | 2 |

Kết quả `nums[2:5] = [5, 2, 3]`.

In [11]:
def trace_largest(nums: List[int], k: int) -> List[int]:
    n = len(nums)
    print(f"nums = {nums}, k = {k}, điểm bắt đầu hợp lệ: 0..{n - k}")
    best_i = 0
    print(f"  khởi tạo best_i = 0 (nums[0] = {nums[0]})")
    for i in range(1, n):
        if i > n - k:
            print(f"  i = {i}: nums[i] = {nums[i]} -> BỎ QUA (không đủ {k} phần tử phía sau)")
            continue
        if nums[i] > nums[best_i]:
            print(f"  i = {i}: {nums[i]} > {nums[best_i]} -> best_i = {i}")
            best_i = i
        else:
            print(f"  i = {i}: {nums[i]} <= {nums[best_i]} -> giữ best_i = {best_i}")
    ans = nums[best_i:best_i + k]
    print("  kết quả:", ans)
    return ans

trace_largest([1, 4, 5, 2, 3], 3)
trace_largest([1, 4, 5, 2, 3], 4)

nums = [1, 4, 5, 2, 3], k = 3, điểm bắt đầu hợp lệ: 0..2
  khởi tạo best_i = 0 (nums[0] = 1)
  i = 1: 4 > 1 -> best_i = 1
  i = 2: 5 > 4 -> best_i = 2
  i = 3: nums[i] = 2 -> BỎ QUA (không đủ 3 phần tử phía sau)
  i = 4: nums[i] = 3 -> BỎ QUA (không đủ 3 phần tử phía sau)
  kết quả: [5, 2, 3]
nums = [1, 4, 5, 2, 3], k = 4, điểm bắt đầu hợp lệ: 0..1
  khởi tạo best_i = 0 (nums[0] = 1)
  i = 1: 4 > 1 -> best_i = 1
  i = 2: nums[i] = 5 -> BỎ QUA (không đủ 4 phần tử phía sau)
  i = 3: nums[i] = 2 -> BỎ QUA (không đủ 4 phần tử phía sau)
  i = 4: nums[i] = 3 -> BỎ QUA (không đủ 4 phần tử phía sau)
  kết quả: [4, 5, 2, 3]


[4, 5, 2, 3]

## 7. Các lời giải khác

### 7.1. Một dòng với `max` + `index`

In [12]:
def largest_index(nums: List[int], k: int) -> List[int]:
    i = nums.index(max(nums[:len(nums) - k + 1]))
    return nums[i:i + k]

print(largest_index([1, 4, 5, 2, 3], 3), largest_index([1, 4, 5, 2, 3], 4))

[5, 2, 3] [4, 5, 2, 3]


### 7.2. `max` trên `range` với `key`

Không tạo lát cắt phụ `nums[:n-k+1]`; tìm thẳng chỉ số.

In [13]:
def largest_key(nums: List[int], k: int) -> List[int]:
    i = max(range(len(nums) - k + 1), key=nums.__getitem__)   # nums.__getitem__(i) == nums[i]
    return nums[i:i + k]

print(largest_key([1, 4, 5, 2, 3], 1), largest_key([1, 4, 5, 2, 3], 5))

[5] [1, 4, 5, 2, 3]


`nums.__getitem__` chính là hàm mà Python gọi khi viết `nums[i]`; dùng nó làm `key` tương đương `lambda i: nums[i]`.

### 7.3. Follow-up — các số **không** phân biệt

Khi có số trùng, hai subarray có thể bằng nhau ở phần tử đầu → phải so tiếp. Ý tưởng:
1. Chỉ các vị trí có `nums[i] == M` (với `M = max(nums[0..n-k])`) mới có thể là đáp án.
2. So sánh các ứng viên đó với nhau bằng so sánh từ điển (dừng ở vị trí khác đầu tiên).

Worst case (ví dụ mảng toàn số bằng nhau) vẫn `O(n·k)`. Có thuật toán `O(n)` (giống bài tìm **hậu tố lớn nhất / xoay vòng nhỏ nhất** bằng hai con trỏ), nhưng vượt phạm vi bài Easy này.

In [14]:
def largest_with_duplicates(nums: List[int], k: int) -> List[int]:
    n = len(nums)
    M = max(nums[:n - k + 1])
    best = None
    for i in range(n - k + 1):
        if nums[i] != M:
            continue                         # chắc chắn thua ngay phần tử đầu
        if best is None:
            best = i
            continue
        for d in range(k):                   # so sánh từ điển với ứng viên tốt nhất
            if nums[i + d] != nums[best + d]:
                if nums[i + d] > nums[best + d]:
                    best = i
                break
    return nums[best:best + k]

print(largest_with_duplicates([5, 1, 5, 3, 5, 2], 3))   # ứng viên bắt đầu bằng 5: [5,1,5], [5,3,5], [5,2] (không hợp lệ)
print(largest_with_duplicates([2, 2, 2, 2], 2))
print(largest_with_duplicates([9, 1, 9, 1, 9, 2], 2))

[5, 3, 5]
[2, 2]
[9, 2]


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space phụ | Dùng được khi có số trùng? |
|---|---|---|---|---|
| Brute `is_larger` | so mọi subarray | `O((n-k+1)·k)` | `O(k)` | ✓ |
| Brute `max(lát cắt)` | so sánh list sẵn của Python | `O((n-k+1)·k)` | `O(k)` | ✓ |
| **Tham lam `best_i`** | max trong `nums[0..n-k]` | `O(n)` | `O(1)` (+ `O(k)` output) | ✗ (chỉ đúng khi phân biệt) |
| `max` + `index` | như trên, một dòng | `O(n)` | `O(n)` lát cắt | ✗ |
| Ứng viên + so sánh | lọc theo max rồi so tiếp | `O(n·k)` worst | `O(1)` | ✓ |

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def largestSubarray(self, nums: List[int], k: int) -> List[int]:
        best_i = 0
        for i in range(1, len(nums) - k + 1):
            if nums[i] > nums[best_i]:
                best_i = i
        return nums[best_i:best_i + k]

sol = Solution()
print(sol.largestSubarray([1, 4, 5, 2, 3], 3), sol.largestSubarray([1, 4, 5, 2, 3], 4), sol.largestSubarray([1, 4, 5, 2, 3], 1))

[5, 2, 3] [4, 5, 2, 3] [5]


Bản ngắn gọn:

In [16]:
class SolutionShort:
    def largestSubarray(self, nums: List[int], k: int) -> List[int]:
        i = nums.index(max(nums[:len(nums) - k + 1]))
        return nums[i:i + k]

print(SolutionShort().largestSubarray([1, 4, 5, 2, 3], 3))

[5, 2, 3]


## 9. Vì sao đúng?

Gọi `S_i = nums[i : i + k]` với `0 ≤ i ≤ n - k` (đúng các subarray dài `k`).

**Bổ đề:** với `i != j`, `S_i > S_j ⇔ nums[i] > nums[j]`.
*Chứng minh:* phần tử đầu tiên của `S_i`, `S_j` là `nums[i]`, `nums[j]`; vì các số phân biệt nên chúng khác nhau ⇒ chỉ số khác nhau đầu tiên là 0 ⇒ theo định nghĩa, so sánh hai mảng chính là so sánh `nums[i]` với `nums[j]`.

**Invariant của vòng lặp:** sau khi xét `i = 0..t`, `best_i` là chỉ số của phần tử lớn nhất trong `nums[0..t]` ⇒ theo bổ đề, `S_best_i` là lớn nhất trong `S_0..S_t`. Khi `t = n - k`, ta có subarray lớn nhất trong **tất cả** subarray dài `k`.

**Vì sao không xét `i > n - k`:** `nums[i:i+k]` khi đó ngắn hơn `k` — không phải subarray dài `k`, không được là đáp án.

## 10. Độ phức tạp

`n = len(nums)`.
- **Tham lam:** duyệt `n - k + 1` phần tử ⇒ `O(n - k + 1) ⊆ O(n)`; tạo kết quả dài `k` ⇒ `O(k)`. **Tổng `O(n)`**.
- **Space:** `O(1)` phụ (không tính mảng kết quả dài `k` bắt buộc phải trả về).
- **Brute force:** `O((n - k + 1) · k)`, tệ nhất `O(n²)` khi `k ≈ n/2`.

In [17]:
import random, time

random.seed(1708)
n = 4000
big = random.sample(range(1, 10**9), n)
k = n // 2
for name, f in [("brute max", largest_brute_max), ("greedy", largest_greedy), ("index", largest_index)]:
    t = time.perf_counter()
    r = f(big, k)
    print(f"{name:>10}: phần tử đầu = {r[0]}, thời gian = {time.perf_counter() - t:.4f}s")

 brute max: phần tử đầu = 998099360, thời gian = 0.0149s
    greedy: phần tử đầu = 998099360, thời gian = 0.0001s
     index: phần tử đầu = 998099360, thời gian = 0.0001s


## 11. Bộ test

1. Ba ví dụ (k = 3, 4, 1).
2. Biên: `n = 1`; `k = n` (đáp án là cả mảng); max nằm ở **cuối** mảng nhưng `k > 1`.
3. Phản ví dụ cho "lấy max cả mảng": max nằm trong `k - 1` phần tử cuối.
4. Đối chiếu brute force và các cách tham lam trên dữ liệu ngẫu nhiên **phân biệt**; đối chiếu cách follow-up với brute trên dữ liệu **có trùng**.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề k=3", S.largestSubarray([1, 4, 5, 2, 3], 3), [5, 2, 3])
check("đề k=4", S.largestSubarray([1, 4, 5, 2, 3], 4), [4, 5, 2, 3])
check("đề k=1", S.largestSubarray([1, 4, 5, 2, 3], 1), [5])
check("n = 1", S.largestSubarray([7], 1), [7])
check("k = n", S.largestSubarray([3, 1, 2], 3), [3, 1, 2])
check("max ở cuối, k = 2", S.largestSubarray([2, 1, 9], 2), [2, 1])
check("max ở đầu", S.largestSubarray([9, 1, 2, 3], 2), [9, 1])

PASS đề k=3: [5, 2, 3]
PASS đề k=4: [4, 5, 2, 3]
PASS đề k=1: [5]
PASS n = 1: [7]
PASS k = n: [3, 1, 2]
PASS max ở cuối, k = 2: [2, 1]
PASS max ở đầu: [9, 1]


In [19]:
random.seed(42)
for _ in range(2000):
    n = random.randint(1, 12)
    nums = random.sample(range(1, 50), n)
    k = random.randint(1, n)
    expected = largest_brute(nums, k)
    for f in [largest_brute_max, largest_greedy, largest_index, largest_key,
              largest_with_duplicates, Solution().largestSubarray, SolutionShort().largestSubarray]:
        assert f(nums, k) == expected, (f.__name__, nums, k)
for _ in range(2000):
    n = random.randint(1, 12)
    nums = [random.randint(1, 3) for _ in range(n)]          # nhiều số trùng
    k = random.randint(1, n)
    assert largest_with_duplicates(nums, k) == largest_brute_max(nums, k), (nums, k)
print("Đối chiếu trên dữ liệu phân biệt và dữ liệu có trùng: PASS")

Đối chiếu trên dữ liệu phân biệt và dữ liệu có trùng: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Lấy max của **cả mảng**

In [20]:
def wrong_global_max(nums, k):
    i = nums.index(max(nums))           # SAI: có thể quá gần cuối
    return nums[i:i + k]

print("sai:", wrong_global_max([1, 4, 5, 2, 3], 4), "| đúng:", largest_greedy([1, 4, 5, 2, 3], 4))

sai: [5, 2, 3] | đúng: [4, 5, 2, 3]


Kết quả sai chỉ dài 3 thay vì 4. **Sửa:** chỉ tìm max trong `nums[:n - k + 1]`.

### Lỗi 2 — Lệch 1 ở biên `range`

Viết `range(n - k)` (thiếu `+ 1`) sẽ bỏ sót điểm bắt đầu cuối cùng `n - k`.

In [21]:
def wrong_off_by_one(nums, k):
    best_i = 0
    for i in range(1, len(nums) - k):    # SAI: thiếu + 1
        if nums[i] > nums[best_i]:
            best_i = i
    return nums[best_i:best_i + k]

print("sai:", wrong_off_by_one([1, 4, 5, 2, 3], 3), "| đúng:", largest_greedy([1, 4, 5, 2, 3], 3))

sai: [4, 5, 2] | đúng: [5, 2, 3]


**Sửa:** `range(1, n - k + 1)`. Mẹo kiểm tra: với `k = n` phải còn đúng một điểm bắt đầu (`i = 0`); với `k = 1` phải xét mọi `i`.

### Lỗi 3 — Hiểu "lớn nhất" là **tổng lớn nhất**

Đề so sánh **từ điển**, không phải tổng. `[5, 1, 1]` lớn hơn `[4, 9, 9]` dù tổng nhỏ hơn.

In [22]:
def wrong_max_sum(nums, k):
    n = len(nums)
    i = max(range(n - k + 1), key=lambda i: sum(nums[i:i + k]))   # SAI: so tổng
    return nums[i:i + k]

nums = [5, 1, 2, 4, 9, 8]
print("sai (tổng):", wrong_max_sum(nums, 3), "| đúng (từ điển):", largest_greedy(nums, 3))

sai (tổng): [4, 9, 8] | đúng (từ điển): [5, 1, 2]


**Sửa:** đọc kỹ định nghĩa so sánh của đề; với số phân biệt chỉ cần phần tử đầu.

### Lỗi 4 — Dùng lời giải tham lam cho dữ liệu có trùng

Nếu đề đổi (follow-up) cho phép trùng, chỉ so phần tử đầu là **không đủ**: `[9, 1, 9, 2]`, `k = 2` → tham lam trả `[9, 1]` nhưng `[9, 2]` lớn hơn.

In [23]:
print("tham lam:", largest_greedy([9, 1, 9, 2], 2), "| đúng khi có trùng:", largest_with_duplicates([9, 1, 9, 2], 2))

tham lam: [9, 1] | đúng khi có trùng: [9, 2]


## 13. Mẹo nhận dạng pattern

- "Lớn nhất theo **thứ tự từ điển**" → nghĩ ngay: **phần tử đầu tiên** quan trọng nhất ⇒ tham lam chọn phần tử đầu lớn nhất có thể.
- "Subarray dài cố định `k`" → điểm bắt đầu hợp lệ `0..n-k` (đếm `n - k + 1`); cẩn thận lệch 1.
- Có điều kiện "**các phần tử phân biệt**" trong đề → thường là gợi ý để đơn giản hoá (không cần xử lý hoà).
- Bài liên quan: 1673 Find the Most Competitive Subsequence (nhỏ nhất từ điển, có stack), 402 Remove K Digits, 1163 Last Substring in Lexicographical Order (hậu tố lớn nhất — follow-up nâng cao), 239 Sliding Window Maximum.

## 14. Biến thể

### Biến thể 1 — Subarray dài `k` **nhỏ nhất** (từ điển)

In [24]:
def smallest_subarray_k(nums, k):
    i = min(range(len(nums) - k + 1), key=lambda i: nums[i])
    return nums[i:i + k]

print(smallest_subarray_k([4, 1, 5, 2, 3], 3), smallest_subarray_k([4, 1, 5, 2, 3], 4))

[1, 5, 2] [1, 5, 2, 3]


### Biến thể 2 — Chuỗi con dài `k` lớn nhất (string thay cho list)

Chuỗi cũng so sánh theo thứ tự từ điển trong Python; khi ký tự có thể trùng, dùng `max` trên các lát cắt (brute force) cho dữ liệu nhỏ.

In [25]:
def largest_substring_k(s, k):
    return max(s[i:i + k] for i in range(len(s) - k + 1))

print(largest_substring_k("leetcode", 3), "|", largest_substring_k("abcabd", 2))

tco | ca


### Biến thể 3 — Trả về **chỉ số bắt đầu** thay vì mảng

In [26]:
def largest_start_index(nums, k):
    best_i = 0
    for i in range(1, len(nums) - k + 1):
        if nums[i] > nums[best_i]:
            best_i = i
    return best_i

print(largest_start_index([1, 4, 5, 2, 3], 3), largest_start_index([1, 4, 5, 2, 3], 4))

2 1


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — So sánh bằng tay
Mảng nào lớn hơn: `[3, 9, 1]` hay `[3, 8, 99]`? `[2, 5]` hay `[10, 0]`? Trả về tuple 2 giá trị `"A"` hoặc `"B"` (A là mảng đứng trước).

In [27]:
def exercise_1():
    # TODO
    pass

# assert exercise_1() == ("A", "B")

### Bài 2 — Viết `is_larger` không dùng `zip`

In [28]:
def is_larger_index(A: List[int], B: List[int]) -> bool:
    # TODO: duyệt i trong range(len(A))
    pass

# assert is_larger_index([1, 3, 2, 4], [1, 2, 2, 4]) == True
# assert is_larger_index([1, 4, 4, 4], [2, 1, 1, 1]) == False

### Bài 3 — Điền chỗ trống lời giải tham lam

In [29]:
def greedy_fill(nums: List[int], k: int) -> List[int]:
    best_i = 0
    # TODO 1: duyệt i trong vùng bắt đầu hợp lệ
    # TODO 2: cập nhật best_i nếu nums[i] lớn hơn
    # TODO 3: trả lát cắt dài k
    pass

# assert greedy_fill([1, 4, 5, 2, 3], 3) == [5, 2, 3]

### Bài 4 — Số lượng subarray dài `k` có phần tử đầu là max của nó
Đếm số `i` hợp lệ sao cho `nums[i] == max(nums[i:i+k])`.

In [30]:
def count_head_is_max(nums: List[int], k: int) -> int:
    # TODO
    pass

# assert count_head_is_max([1, 4, 5, 2, 3], 3) == 1
# assert count_head_is_max([5, 4, 3, 2, 1], 2) == 4

### Bài 5 — Biến thể: tổng lớn nhất của subarray dài `k`
Trả về **tổng** lớn nhất (không phải theo từ điển) trong `O(n)` bằng **cửa sổ trượt**.

In [31]:
def max_sum_window(nums: List[int], k: int) -> int:
    # TODO: tổng cửa sổ đầu, rồi cộng phần tử vào, trừ phần tử ra
    pass

# assert max_sum_window([5, 1, 2, 4, 9, 8], 3) == 21

### Bài 6 — Mini challenge: follow-up nhanh hơn trong thực tế
Viết `largest_dup_fast(nums, k)` cho dữ liệu **có trùng**, dùng phép so sánh list của Python giữa các ứng viên có phần tử đầu bằng max (không tự viết vòng `for d`).

In [32]:
def largest_dup_fast(nums: List[int], k: int) -> List[int]:
    # TODO
    pass

# assert largest_dup_fast([9, 1, 9, 2], 2) == [9, 2]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Tìm vị trí khác đầu tiên. (2) Cặp 1: vị trí 1 (9 vs 8); cặp 2: vị trí 0 (2 vs 10). (3) `("A", "B")`.

**Bài 2:** (1) Giống `is_larger`. (2) `for i in range(len(A)): if A[i] != B[i]: return A[i] > B[i]`. (3) Cuối `return False`.

**Bài 3:** (1) `range(1, len(nums) - k + 1)`. (2) `if nums[i] > nums[best_i]: best_i = i`. (3) `return nums[best_i:best_i + k]`.

**Bài 4:** (1) Duyệt `i` hợp lệ. (2) Dùng `max(nums[i:i+k])`. (3) `sum(nums[i] == max(nums[i:i+k]) for i in range(n-k+1))`.

**Bài 5:** (1) `window = sum(nums[:k])`. (2) Với `i` từ `k`: `window += nums[i] - nums[i-k]`. (3) Giữ `best = max(best, window)`.

**Bài 6:** (1) `M = max(nums[:n-k+1])`. (2) Ứng viên là `nums[i:i+k]` với `nums[i] == M`. (3) `max(ứng_viên)`.

## 17. Đáp án tham khảo

In [33]:
def ans_exercise_1():
    return ("A" if [3, 9, 1] > [3, 8, 99] else "B", "A" if [2, 5] > [10, 0] else "B")

def ans_is_larger_index(A, B):
    for i in range(len(A)):
        if A[i] != B[i]:
            return A[i] > B[i]
    return False

def ans_greedy_fill(nums, k):
    best_i = 0
    for i in range(1, len(nums) - k + 1):
        if nums[i] > nums[best_i]:
            best_i = i
    return nums[best_i:best_i + k]

def ans_count_head_is_max(nums, k):
    return sum(nums[i] == max(nums[i:i + k]) for i in range(len(nums) - k + 1))

def ans_max_sum_window(nums, k):
    window = sum(nums[:k])
    best = window
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]
        best = max(best, window)
    return best

def ans_largest_dup_fast(nums, k):
    n = len(nums)
    M = max(nums[:n - k + 1])
    return max(nums[i:i + k] for i in range(n - k + 1) if nums[i] == M)

assert ans_exercise_1() == ("A", "B")
assert ans_is_larger_index([1, 3, 2, 4], [1, 2, 2, 4]) == True
assert ans_is_larger_index([1, 4, 4, 4], [2, 1, 1, 1]) == False
assert ans_greedy_fill([1, 4, 5, 2, 3], 3) == [5, 2, 3]
assert ans_count_head_is_max([1, 4, 5, 2, 3], 3) == 1
assert ans_count_head_is_max([5, 4, 3, 2, 1], 2) == 4
assert ans_max_sum_window([5, 1, 2, 4, 9, 8], 3) == 21
assert ans_largest_dup_fast([9, 1, 9, 2], 2) == [9, 2]
for _ in range(1000):
    n = random.randint(1, 10)
    nums = [random.randint(1, 4) for _ in range(n)]
    k = random.randint(1, n)
    assert ans_largest_dup_fast(nums, k) == largest_brute_max(nums, k)
    assert ans_max_sum_window(nums, k) == max(sum(nums[i:i + k]) for i in range(n - k + 1))
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def largestSubarray(self, nums: List[int], k: int) -> List[int]:
        best_i = 0
        for i in range(1, len(nums) - k + 1):   # chỉ điểm bắt đầu hợp lệ 0..n-k
            if nums[i] > nums[best_i]:          # số phân biệt -> phần tử đầu quyết định
                best_i = i
        return nums[best_i:best_i + k]
```

**Khung tư duy:** so sánh từ điển → phần tử đầu quan trọng nhất → tham lam chọn đầu lớn nhất **trong vùng hợp lệ**.

**Checklist trước khi submit**
- [ ] Vùng bắt đầu là `0..n-k` (`range(..., n - k + 1)`)?
- [ ] Không lấy max của cả mảng?
- [ ] Hiểu "lớn nhất" là theo **từ điển**, không phải tổng?
- [ ] Đã thử `k = 1`, `k = n`, max nằm ở cuối?
- [ ] Nếu đề cho phép số trùng: có so sánh tiếp sau phần tử đầu?
- [ ] Tên method `largestSubarray`, trả về `List[int]` dài đúng `k`?